# OCR Corrector — Fine-tuning en Google Colab (T4)

**Modelo:** `google/flan-t5-base`  
**Tarea:** seq2seq — `correct OCR: {texto_ocr}` → `{texto_limpio}`  
**Dataset:** 1728 pares OCR↔abstract (1382 train / 172 val / 174 test)

> **Antes de empezar:** `Entorno de ejecución → Cambiar tipo → GPU (T4)`

## 1. Verificar GPU

In [ ]:
import torch
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2. Clonar repositorio e instalar dependencias

In [ ]:
!git clone https://github.com/Cronosteak/ocr-corrector-papers.git
%cd ocr-corrector-papers

In [ ]:
!pip install -q transformers datasets pyyaml accelerate

## 3. Subir dataset desde local

Los pares no están en el repo (data/ está en .gitignore). Subir `data/pairs/train.json`, `val.json` y `test.json` aquí:

In [ ]:
from google.colab import files
import os

os.makedirs("data/pairs", exist_ok=True)
uploaded = files.upload()  # seleccionar train.json, val.json, test.json

for name, content in uploaded.items():
    with open(f"data/pairs/{name}", "wb") as f:
        f.write(content)
    print(f"Guardado: data/pairs/{name}")

In [ ]:
import json
for split in ["train", "val", "test"]:
    d = json.load(open(f"data/pairs/{split}.json"))
    print(f"{split}: {len(d)} pares")

## 4. Entrenar

In [ ]:
# fp16=true para aprovechar la T4
import yaml

with open("configs/train_config.yaml") as f:
    config = yaml.safe_load(f)

config["fp16"] = True

with open("configs/train_config.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("fp16 activado para T4")
print(yaml.dump(config, default_flow_style=False))

In [ ]:
!python -m src.model.train

## 5. Evaluar (CER / WER baseline vs corregido)

In [ ]:
!pip install -q jiwer

In [ ]:
!python -m src.model.evaluate --model models/ocr-corrector --data data/pairs/test.json

## 6. Ejemplos cualitativos

In [ ]:
import json
import random
from src.model.predict import load_model, correct_text

test_pairs = json.load(open("data/pairs/test.json"))
model, tokenizer = load_model("models/ocr-corrector")

samples = random.sample(test_pairs, 5)
for i, pair in enumerate(samples, 1):
    corrected = correct_text(pair["ocr"], model=model, tokenizer=tokenizer)
    print(f"--- Ejemplo {i} ---")
    print(f"OCR:       {pair['ocr'][:120]}")
    print(f"Corregido: {corrected[:120]}")
    print(f"GT:        {pair['ground_truth'][:120]}")
    print()

## 7. Descargar modelo entrenado

In [ ]:
!zip -r ocr-corrector-model.zip models/ocr-corrector/
from google.colab import files
files.download("ocr-corrector-model.zip")